# YouTube GraphRAG Embedding Pipeline

End-to-end test harness for the AI chat app. This notebook:

1. Reads YouTube video docs from MongoDB (`youtube_channel_videos`).
2. Upserts them into Neo4j as:
   - `(:YouTubeVideo { video_id, video_title, video_description, video_url, published_at, view_count, like_count, comment_count, thumbnail_url, thumbnail_description, thumbnail_keywords, comment_summary_description, ... })`  (assumes the node and its `(:YouTubeChannel)-[:HAS_VIDEO]->(:YouTubeVideo)` link were already created by the prod sync)
   - `(:YouTubeCommentTopic {video_id, name, category, platform})`
   - `(v)-[:HAS_COMMENT_TOPIC {weight, platform: 'youtube'}]->(t)`
3. Computes Azure OpenAI embeddings for three distinct signals:
   - `Topic.embedding` (from `Topic.name`) – comment-derived themes.
   - `YouTubeVideo.comment_summary_embedding` (from `comment_summary_description`) – what the comment section is talking about.
   - `YouTubeVideo.video_content_embedding` (from title + description + thumbnail_description + thumbnail_keywords + tags) – what the **video itself** is about / shows.
4. Creates Neo4j vector indexes on all three fields.
5. Runs example semantic queries: top creators / example videos / comment-section discussions / content-imagery search / fused unified search.

Each step is **idempotent and resumable** – embeddings are only recomputed for nodes whose source text has changed or is missing.

> Note on geographic / demographic questions (e.g. *"in which neighborhoods..."*): we don't currently store any location or audience-demographic data on videos or channels, so those dimensions can't be answered from this graph alone. The pipeline below covers every text signal we *do* have.


In [26]:
import os
import json
import time
from typing import Iterable, Iterator, List, Dict, Any, Optional

from dotenv import load_dotenv
from pymongo import MongoClient
from neo4j import GraphDatabase
from openai import AzureOpenAI, OpenAI

load_dotenv(os.getenv('DOTENV_PATH'), override=True) if os.getenv('DOTENV_PATH') else load_dotenv(override=True)
print('Env loaded from', os.getenv('DOTENV_PATH') or 'default search path')

Env loaded from /home/jovyan/.env


In [27]:
# ---- Mongo (Azure Cosmos DB for Mongo) ----
def _redact_mongo_uri(uri: str) -> str:
    """Hide user:password in mongodb://... for safe logging / notebook output."""
    if not uri or '://' not in uri:
        return uri
    scheme, rest = uri.split('://', 1)
    if '@' not in rest:
        return uri
    creds, hostpart = rest.rsplit('@', 1)
    return f'{scheme}://***:***@{hostpart}'

MONGO_URI = os.getenv('MONGODB_URI') or os.getenv('MONGO_URI', 'mongodb://localhost:27017')
MONGO_DB = os.getenv('MONGODB_DB') or os.getenv('MONGO_DB', 'rbl')
MONGO_COLLECTION = os.getenv('MONGO_COLLECTION', 'youtube_channel_videos')

# ---- Neo4j (local dev) ----
def _neo4j_uri_for_runtime(uri: str) -> str:
    """Rewrite Docker-only hostnames (`neo4j`, `host.docker.internal`) to localhost when the kernel runs on the Mac host."""
    if not uri or '://' not in uri:
        return uri
    from urllib.parse import urlparse
    p = urlparse(uri)
    host = (p.hostname or '').lower()
    if host not in {'neo4j', 'host.docker.internal'}:
        return uri
    import socket
    try:
        socket.getaddrinfo(p.hostname, p.port or 7687)
    except OSError:
        netloc = f'localhost:{p.port or 7687}'
        return p._replace(netloc=netloc).geturl()
    return uri


NEO4J_URI = _neo4j_uri_for_runtime(
    os.getenv('NEO4J_URI_DEV') or os.getenv('NEO4J_URI', 'bolt://neo4j:7687')
)
NEO4J_USER = os.getenv('NEO4J_USER_DEV') or os.getenv('NEO4J_USER', 'neo4j')
NEO4J_PASSWORD = os.getenv('NEO4J_PASSWORD_DEV') or os.getenv('NEO4J_PASSWORD', 'neo4j')
NEO4J_DATABASE = os.getenv('NEO4J_DATABASE', 'neo4j')

# ---- Azure OpenAI — EMBEDDINGS resource ----
# Embeddings live on a *separate* Azure resource from chat completions,
# with its own endpoint, key and API version. Only these are used in this notebook.
AZURE_OPENAI_EMBEDDING_ENDPOINT = os.getenv('AZURE_OPENAI_EMBEDDING_ENDPOINT')
AZURE_OPENAI_EMBEDDING_API_KEY = os.getenv('AZURE_OPENAI_EMBEDDING_API_KEY')
AZURE_OPENAI_EMBEDDING_API_VERSION = os.getenv('AZURE_OPENAI_EMBEDDING_API_VERSION', '2024-02-01')
AZURE_OPENAI_EMBEDDING_DEPLOYMENT = (
    os.getenv('AZURE_OPENAI_EMBEDDING_DEPLOYMENT_MODEL_NAME')
    or os.getenv('AZURE_OPENAI_EMBEDDING_DEPLOYMENT_NAME')
    or os.getenv('AZURE_OPENAI_EMBEDDING_DEPLOYMENT')
    or 'text-embedding-3-large'
)

# Fallback: vanilla OpenAI
OPENAI_API_KEY = os.getenv('OPENAI_API_KEY')
OPENAI_EMBEDDING_MODEL = os.getenv('OPENAI_EMBEDDING_MODEL', 'text-embedding-3-large')

# ---- Batch sizes ----
MONGO_FETCH_LIMIT = int(os.getenv('MONGO_FETCH_LIMIT', '0'))          # 0 = no limit
VIDEO_UPSERT_BATCH = int(os.getenv('VIDEO_UPSERT_BATCH', '200'))
EMBED_BATCH = int(os.getenv('EMBED_BATCH', '128'))                    # items per embedding API call
EMBED_WRITE_BATCH = int(os.getenv('EMBED_WRITE_BATCH', '500'))        # items per Neo4j write tx

YOUTUBE_PLATFORM = 'youtube'  # HAS_COMMENT_TOPIC / YouTubeCommentTopic.platform (matches prod notebook)

print('Config:')
print(f'  mongo: {_redact_mongo_uri(MONGO_URI)}  db={MONGO_DB}  collection={MONGO_COLLECTION}')
print(f'  neo4j: {NEO4J_URI}  db={NEO4J_DATABASE}  user={NEO4J_USER}')
print(f'  embeddings: azure_endpoint={AZURE_OPENAI_EMBEDDING_ENDPOINT}  azure_deployment={AZURE_OPENAI_EMBEDDING_DEPLOYMENT}  azure_api_version={AZURE_OPENAI_EMBEDDING_API_VERSION}  openai_fallback={OPENAI_EMBEDDING_MODEL}')
print(f'  batches: mongo_limit={MONGO_FETCH_LIMIT}  upsert={VIDEO_UPSERT_BATCH}  embed={EMBED_BATCH}  write={EMBED_WRITE_BATCH}')


Config:
  mongo: mongodb://***:***@  db=rbl  collection=youtube_channel_videos
  neo4j: bolt://host.docker.internal:7687  db=neo4j  user=neo4j
  embeddings: azure_endpoint=https://rbtl-embedding-resource.cognitiveservices.azure.com/  azure_deployment=text-embedding-3-large  azure_api_version=2024-02-01  openai_fallback=text-embedding-3-large
  batches: mongo_limit=0  upsert=200  embed=128  write=500


In [28]:
mongo_client = MongoClient(MONGO_URI)
mongo_db = mongo_client[MONGO_DB]
mongo_coll = mongo_db[MONGO_COLLECTION]
print('Mongo collection count:', mongo_coll.estimated_document_count())

/tmp/ipykernel_66/2705439973.py:1: UserWarning: You appear to be connected to a CosmosDB cluster. For more information regarding feature compatibility and support please visit https://www.mongodb.com/supportability/cosmosdb
  mongo_client = MongoClient(MONGO_URI)


Mongo collection count: 25803


In [29]:
driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USER, NEO4J_PASSWORD))
with driver.session(database=NEO4J_DATABASE) as s:
    rec = s.run('RETURN 1 AS ok').single()
    print('Neo4j ok:', rec['ok'])
    for label in ['YouTubeVideo', 'YouTubeCommentTopic', 'YouTubeChannel']:
        cnt = s.run(f'MATCH (n:{label}) RETURN count(n) AS c').single()['c']
        print(f'  {label}: {cnt}')

Neo4j ok: 1
  YouTubeVideo: 25803
  YouTubeCommentTopic: 0
  YouTubeChannel: 196


In [30]:
def build_embedding_client():
    """Prefer the dedicated Azure OpenAI embeddings resource, else plain OpenAI."""
    if AZURE_OPENAI_EMBEDDING_ENDPOINT and AZURE_OPENAI_EMBEDDING_API_KEY:
        c = AzureOpenAI(
            azure_endpoint=AZURE_OPENAI_EMBEDDING_ENDPOINT,
            api_key=AZURE_OPENAI_EMBEDDING_API_KEY,
            api_version=AZURE_OPENAI_EMBEDDING_API_VERSION,
        )
        return ('azure', c, AZURE_OPENAI_EMBEDDING_DEPLOYMENT)
    if OPENAI_API_KEY:
        return ('openai', OpenAI(api_key=OPENAI_API_KEY), OPENAI_EMBEDDING_MODEL)
    raise RuntimeError(
        'No embedding credentials: set AZURE_OPENAI_EMBEDDING_ENDPOINT + '
        'AZURE_OPENAI_EMBEDDING_API_KEY (and AZURE_OPENAI_EMBEDDING_DEPLOYMENT_MODEL_NAME), '
        'or OPENAI_API_KEY.'
    )

EMBED_PROVIDER, embed_client, EMBED_MODEL_NAME = build_embedding_client()
print(f'Embedding provider: {EMBED_PROVIDER}  model/deployment: {EMBED_MODEL_NAME}')

probe = embed_client.embeddings.create(model=EMBED_MODEL_NAME, input=['ping'])
EMBED_DIM = len(probe.data[0].embedding)
print(f'Embedding dimensions: {EMBED_DIM}')

Embedding provider: azure  model/deployment: text-embedding-3-large
Embedding dimensions: 3072


In [24]:
def batched(iterable: Iterable, n: int) -> Iterator[list]:
    batch = []
    for item in iterable:
        batch.append(item)
        if len(batch) >= n:
            yield batch
            batch = []
    if batch:
        yield batch


def embed_texts(texts: List[str], max_retries: int = 5) -> List[List[float]]:
    """Call the embedding API with simple exponential-backoff retries."""
    out: List[List[float]] = []
    for chunk in batched(texts, EMBED_BATCH):
        attempt = 0
        while True:
            try:
                resp = embed_client.embeddings.create(model=EMBED_MODEL_NAME, input=chunk)
                out.extend([d.embedding for d in resp.data])
                break
            except Exception as e:
                attempt += 1
                if attempt > max_retries:
                    raise
                wait = min(2 ** attempt, 30)
                print(f'  embed retry {attempt} after {wait}s: {e}')
                time.sleep(wait)
    return out


def clean_text(value: Any) -> Optional[str]:
    """Mongo string fields are passed through with no strip and no truncation (byte-exact vs driver)."""
    if value is None:
        return None
    if isinstance(value, str):
        return value
    return str(value)


def _iso(v):
    if v is None:
        return None
    if hasattr(v, 'isoformat'):
        return v.isoformat()
    return str(v)


def build_video_content_text(
    title: Optional[str],
    description: Optional[str],
    thumbnail_description: Optional[str],
    thumbnail_keywords: List[str],
    tags: List[str],
) -> Optional[str]:
    """Combine all video-side text signals into one document to embed (full text, no truncation)."""
    parts: List[str] = []
    if title:
        parts.append(f'Title: {title}')
    if description:
        parts.append(f'Description: {description}')
    if thumbnail_description:
        parts.append(f'Thumbnail: {thumbnail_description}')
    if thumbnail_keywords:
        parts.append(f'Thumbnail keywords: {", ".join(thumbnail_keywords)}')
    if tags:
        parts.append(f'Tags: {", ".join(tags)}')
    if not parts:
        return None
    return '\n\n'.join(parts)


def normalize_mongo_doc(doc: dict) -> Dict[str, Any]:
    """Flatten one Mongo doc into a Neo4j-ready dict."""
    raw_video_id = doc.get('video_id')
    if raw_video_id is None:
        return {}
    video_id = str(raw_video_id).strip()
    if not video_id:
        return {}

    topics = doc.get('comments_frequent_topics') or []
    categories = doc.get('comments_frequent_topic_categories') or []
    weights = doc.get('comments_frequent_topic_weights') or []

    topic_rows = []
    for i, name in enumerate(topics):
        name = clean_text(name)
        if not name:
            continue
        topic_rows.append({
            'name': name,
            'category': clean_text(categories[i] if i < len(categories) else None),
            'weight': float(weights[i]) if i < len(weights) and weights[i] is not None else None,
            'position': i,
        })

    video_title = clean_text(doc.get('video_title'))
    video_description = clean_text(doc.get('video_description'))
    thumbnail_description = clean_text(doc.get('thumbnail_description'))
    thumbnail_keywords = [k for k in (doc.get('thumbnail_keywords') or []) if isinstance(k, str)]
    tags = [t for t in (doc.get('tags') or []) if isinstance(t, str)]

    video_content_text = build_video_content_text(
        video_title,
        video_description,
        thumbnail_description,
        thumbnail_keywords,
        tags,
    )

    return {
        'video_id': video_id,
        'video_title': video_title,
        'video_description': video_description,
        'video_url': doc.get('video_url') or f"https://www.youtube.com/watch?v={video_id}",
        'published_at': _iso(doc.get('published_at')),
        'view_count': doc.get('view_count'),
        'like_count': doc.get('like_count'),
        'comment_count': doc.get('comment_count'),
        'thumbnail_url': doc.get('thumbnail_url'),
        'thumbnail_description': thumbnail_description,
        'thumbnail_keywords': thumbnail_keywords,
        'tags': tags,
        'topic_categories': doc.get('topic_categories') or [],
        'defaultLanguage': doc.get('defaultLanguage'),
        'defaultAudioLanguage': doc.get('defaultAudioLanguage'),
        'comment_summary_description': clean_text(doc.get('comment_summary_description')),
        'video_content_text': video_content_text,
        'channel_id': doc.get('channel_id'),
        'channel_title': doc.get('channel_title'),
        'username': doc.get('username'),
        'topics': topic_rows,
    }


In [25]:
def ensure_schema(driver):
    stmts = [
        'CREATE CONSTRAINT video_id_unique IF NOT EXISTS FOR (v:YouTubeVideo) REQUIRE v.video_id IS UNIQUE',
        'CREATE CONSTRAINT youtube_comment_topic_video_name IF NOT EXISTS FOR (t:YouTubeCommentTopic) REQUIRE (t.video_id, t.name) IS UNIQUE',
    ]
    with driver.session(database=NEO4J_DATABASE) as s:
        for stmt in stmts:
            s.run(stmt)
    print('Schema constraints ensured.')

ensure_schema(driver)


Schema constraints ensured.


In [17]:
UPSERT_VIDEO_CYPHER = '''
UNWIND $rows AS row

// Video (channel + HAS_VIDEO already exist from prod sync; we don't touch them)
MERGE (v:YouTubeVideo {video_id: row.video_id})
SET v.video_title                 = row.video_title,
    v.video_description            = row.video_description,
    v.video_url                    = row.video_url,
    v.published_at                 = row.published_at,
    v.view_count                   = row.view_count,
    v.like_count                   = row.like_count,
    v.comment_count                = row.comment_count,
    v.thumbnail_url                = row.thumbnail_url,
    v.thumbnail_description        = row.thumbnail_description,
    v.thumbnail_keywords           = row.thumbnail_keywords,
    v.tags                         = row.tags,
    v.topic_categories             = row.topic_categories,
    v.defaultLanguage              = row.defaultLanguage,
    v.defaultAudioLanguage         = row.defaultAudioLanguage,
    v.channel_id                   = row.channel_id,
    v.channel_title                = row.channel_title,
    v.username                     = row.username

// Keep comment_summary + its embedding in sync; invalidate embedding if text changed.
WITH row, v
FOREACH (_ IN CASE WHEN row.comment_summary_description IS NOT NULL AND row.comment_summary_description <> coalesce(v.comment_summary_description, '') THEN [1] ELSE [] END |
  SET v.comment_summary_description = row.comment_summary_description,
      v.comment_summary_embedding   = NULL
)
FOREACH (_ IN CASE WHEN row.comment_summary_description IS NOT NULL AND v.comment_summary_description IS NULL THEN [1] ELSE [] END |
  SET v.comment_summary_description = row.comment_summary_description
)

// Keep the unified content text + its embedding in sync.
WITH row, v
FOREACH (_ IN CASE WHEN row.video_content_text IS NOT NULL AND row.video_content_text <> coalesce(v.video_content_text, '') THEN [1] ELSE [] END |
  SET v.video_content_text    = row.video_content_text,
      v.video_content_embedding = NULL
)
FOREACH (_ IN CASE WHEN row.video_content_text IS NOT NULL AND v.video_content_text IS NULL THEN [1] ELSE [] END |
  SET v.video_content_text = row.video_content_text
)

// Replace HAS_COMMENT_TOPIC relationships so they stay in sync with Mongo
WITH row, v
OPTIONAL MATCH (v)-[r:HAS_COMMENT_TOPIC]->()
DELETE r

WITH row, v
UNWIND row.topics AS topic
MERGE (t:YouTubeCommentTopic {video_id: row.video_id, name: topic.name})
SET t.category = topic.category,
    t.platform = $youtube_platform
MERGE (v)-[rel:HAS_COMMENT_TOPIC]->(t)
SET rel.weight = topic.weight,
    rel.position = topic.position,
    rel.platform = $youtube_platform
RETURN count(*) AS topic_rows
'''


def upsert_videos_to_neo4j(driver, docs_iter):
    total_videos = 0
    total_topic_rows = 0
    for batch in batched(docs_iter, VIDEO_UPSERT_BATCH):
        rows = [normalize_mongo_doc(d) for d in batch]
        rows = [r for r in rows if r.get('video_id')]
        if not rows:
            continue
        with driver.session(database=NEO4J_DATABASE) as s:
            result = s.run(UPSERT_VIDEO_CYPHER, rows=rows, youtube_platform=YOUTUBE_PLATFORM)
            topic_rows = sum(rec['topic_rows'] for rec in result)
        total_videos += len(rows)
        total_topic_rows += topic_rows
        print(f'  upserted videos={total_videos}  topic_rows={total_topic_rows}')
    return total_videos, total_topic_rows


In [ ]:
MONGO_PROJECTION = {
    'video_id': 1, 'video_title': 1, 'video_description': 1, 'video_url': 1,
    'published_at': 1, 'view_count': 1, 'like_count': 1, 'comment_count': 1,
    'thumbnail_url': 1, 'thumbnail_description': 1, 'thumbnail_keywords': 1,
    'tags': 1, 'topic_categories': 1, 'defaultLanguage': 1, 'defaultAudioLanguage': 1,
    'channel_id': 1, 'channel_title': 1, 'username': 1,
    'comment_summary_description': 1,
    'comments_frequent_topics': 1, 'comments_frequent_topic_categories': 1, 'comments_frequent_topic_weights': 1,
}

query = {'video_id': {'$exists': True}}
cursor = mongo_coll.find(query, MONGO_PROJECTION, no_cursor_timeout=True)
if MONGO_FETCH_LIMIT:
    cursor = cursor.limit(MONGO_FETCH_LIMIT)

start = time.time()
try:
    videos, topic_rows = upsert_videos_to_neo4j(driver, cursor)
finally:
    cursor.close()
print(f'Done. videos={videos}  topic_rows={topic_rows}  took {time.time()-start:.1f}s')

with driver.session(database=NEO4J_DATABASE) as s:
    for q, label in [
        ('MATCH (v:YouTubeVideo) RETURN count(v) AS c', 'YouTubeVideo'),
        ('MATCH (c:YouTubeChannel) RETURN count(c) AS c', 'YouTubeChannel'),
        ('MATCH (t:YouTubeCommentTopic) RETURN count(t) AS c', 'YouTubeCommentTopic'),
        ('MATCH ()-[r:HAS_COMMENT_TOPIC]->() WHERE coalesce(r.platform, \'youtube\') = \'youtube\' RETURN count(r) AS c', 'HAS_COMMENT_TOPIC_youtube'),
        ('MATCH (v:YouTubeVideo) WHERE v.comment_summary_description IS NOT NULL RETURN count(v) AS c', 'videos_with_summary'),
    ]:
        print(f'{label}: {s.run(q).single()["c"]}')


In [ ]:
def ensure_vector_indexes(driver, dim: int):
    stmts = [
        f'''CREATE VECTOR INDEX youtube_comment_topic_embedding_index IF NOT EXISTS
            FOR (t:YouTubeCommentTopic) ON (t.embedding)
            OPTIONS {{ indexConfig: {{ `vector.dimensions`: {dim}, `vector.similarity_function`: 'cosine' }} }}''',
        f'''CREATE VECTOR INDEX video_summary_embedding_index IF NOT EXISTS
            FOR (v:YouTubeVideo) ON (v.comment_summary_embedding)
            OPTIONS {{ indexConfig: {{ `vector.dimensions`: {dim}, `vector.similarity_function`: 'cosine' }} }}''',
        f'''CREATE VECTOR INDEX video_content_embedding_index IF NOT EXISTS
            FOR (v:YouTubeVideo) ON (v.video_content_embedding)
            OPTIONS {{ indexConfig: {{ `vector.dimensions`: {dim}, `vector.similarity_function`: 'cosine' }} }}''',
    ]
    with driver.session(database=NEO4J_DATABASE) as s:
        for stmt in stmts:
            s.run(stmt)
        rows = list(s.run('SHOW VECTOR INDEXES YIELD name, labelsOrTypes, properties, state, options'))
    for r in rows:
        print(dict(r))

ensure_vector_indexes(driver, EMBED_DIM)


In [ ]:
TOPIC_FETCH_CYPHER = '''
MATCH (t:YouTubeCommentTopic)
WHERE coalesce(t.platform, 'youtube') = $platform
  AND t.embedding IS NULL AND t.name IS NOT NULL
RETURN elementId(t) AS eid, t.name AS name
LIMIT $limit
'''

TOPIC_WRITE_CYPHER = '''
UNWIND $rows AS row
MATCH (t) WHERE elementId(t) = row.eid
CALL db.create.setNodeVectorProperty(t, 'embedding', row.embedding)
RETURN count(*) AS written
'''


def embed_topic_nodes(driver, page_size: int = 2000):
    total_written = 0
    start = time.time()
    while True:
        with driver.session(database=NEO4J_DATABASE) as s:
            pending = list(s.run(TOPIC_FETCH_CYPHER, limit=page_size, platform=YOUTUBE_PLATFORM))
        if not pending:
            break
        texts = [r['name'] for r in pending]
        embeddings = embed_texts(texts)
        rows = [
            {'eid': pending[i]['eid'], 'embedding': embeddings[i]}
            for i in range(len(pending))
        ]
        for chunk in batched(rows, EMBED_WRITE_BATCH):
            with driver.session(database=NEO4J_DATABASE) as s:
                written = s.run(TOPIC_WRITE_CYPHER, rows=chunk).single()['written']
            total_written += written
        print(f'  topic embeddings written so far: {total_written}  (elapsed {time.time()-start:.1f}s)')
    print(f'Done. topic embeddings written: {total_written}')

embed_topic_nodes(driver)


In [ ]:
VIDEO_FETCH_CYPHER = '''
MATCH (v:YouTubeVideo)
WHERE v.comment_summary_description IS NOT NULL
  AND v.comment_summary_embedding IS NULL
RETURN elementId(v) AS eid, v.comment_summary_description AS text
LIMIT $limit
'''

VIDEO_WRITE_CYPHER = '''
UNWIND $rows AS row
MATCH (v) WHERE elementId(v) = row.eid
CALL db.create.setNodeVectorProperty(v, 'comment_summary_embedding', row.embedding)
RETURN count(*) AS written
'''


def embed_video_summaries(driver, page_size: int = 500):
    total_written = 0
    start = time.time()
    while True:
        with driver.session(database=NEO4J_DATABASE) as s:
            pending = list(s.run(VIDEO_FETCH_CYPHER, limit=page_size))
        if not pending:
            break
        texts = [r['text'] for r in pending]
        embeddings = embed_texts(texts)
        rows = [
            {'eid': pending[i]['eid'], 'embedding': embeddings[i]}
            for i in range(len(pending))
        ]
        for chunk in batched(rows, EMBED_WRITE_BATCH):
            with driver.session(database=NEO4J_DATABASE) as s:
                written = s.run(VIDEO_WRITE_CYPHER, rows=chunk).single()['written']
            total_written += written
        print(f'  video summary embeddings written so far: {total_written}  (elapsed {time.time()-start:.1f}s)')
    print(f'Done. video summary embeddings written: {total_written}')

embed_video_summaries(driver)


In [ ]:
CONTENT_FETCH_CYPHER = '''
MATCH (v:YouTubeVideo)
WHERE v.video_content_text IS NOT NULL
  AND v.video_content_embedding IS NULL
RETURN elementId(v) AS eid, v.video_content_text AS text
LIMIT $limit
'''

CONTENT_WRITE_CYPHER = '''
UNWIND $rows AS row
MATCH (v) WHERE elementId(v) = row.eid
CALL db.create.setNodeVectorProperty(v, 'video_content_embedding', row.embedding)
RETURN count(*) AS written
'''


def embed_video_content(driver, page_size: int = 500):
    total_written = 0
    start = time.time()
    while True:
        with driver.session(database=NEO4J_DATABASE) as s:
            pending = list(s.run(CONTENT_FETCH_CYPHER, limit=page_size))
        if not pending:
            break
        texts = [r['text'] for r in pending]
        embeddings = embed_texts(texts)
        rows = [
            {'eid': pending[i]['eid'], 'embedding': embeddings[i]}
            for i in range(len(pending))
        ]
        for chunk in batched(rows, EMBED_WRITE_BATCH):
            with driver.session(database=NEO4J_DATABASE) as s:
                written = s.run(CONTENT_WRITE_CYPHER, rows=chunk).single()['written']
            total_written += written
        print(f'  video content embeddings written so far: {total_written}  (elapsed {time.time()-start:.1f}s)')
    print(f'Done. video content embeddings written: {total_written}')

embed_video_content(driver)


## Test queries

These are the retrieval primitives the AI chat app will use. Each takes a theme (e.g. `"vaping"`, `"imagery of weapons or violence"`, `"gambling"`), embeds it, then hits one of the Neo4j vector indexes.

- **`top_creators(theme)`** – vector search over `Topic.embedding`, aggregate weighted contribution per `Channel`. Best for *"which influencers' audiences talk about X"*.
- **`example_videos(theme)`** – vector search over `Topic.embedding`, return videos whose comment-topics match. Best for *"show me examples of X content"* when you want comment-driven evidence.
- **`comment_discussions(theme)`** – vector search over `comment_summary_embedding`. Best for *"whose comment section discusses X"*.
- **`content_videos(theme)`** – vector search over `video_content_embedding` (title + description + thumbnail description + thumbnail keywords + tags). Best for **"which videos show / are about X"**, including visual imagery questions.
- **`content_top_creators(theme)`** – aggregates `content_videos` hits by creator.
- **`unified_search(theme)`** – runs all three vector indexes and fuses results with Reciprocal Rank Fusion (RRF). Use this when you don't know which signal the user's question maps to.

In [ ]:
def embed_query(text: str) -> List[float]:
    return embed_texts([text])[0]


TOP_CREATORS_CYPHER = '''
CALL db.index.vector.queryNodes('youtube_comment_topic_embedding_index', $k, $q) YIELD node AS t, score
WHERE coalesce(t.platform, 'youtube') = $platform
MATCH (v:YouTubeVideo)-[rel:HAS_COMMENT_TOPIC]->(t)
WHERE coalesce(rel.platform, 'youtube') = $platform
OPTIONAL MATCH (c:YouTubeChannel)-[:HAS_VIDEO]->(v)
WITH
  coalesce(c.channel_title, v.channel_title, v.username, 'unknown') AS creator,
  coalesce(c.channel_id, v.channel_id) AS channel_id,
  t, v, rel, score
WITH
  creator,
  channel_id,
  sum(score * coalesce(rel.weight, 1.0)) AS relevance,
  count(DISTINCT v) AS video_count,
  collect(DISTINCT t.name)[0..5] AS sample_topics,
  collect(DISTINCT {video_id: v.video_id, title: v.video_title, url: v.video_url})[0..8] AS sample_videos
RETURN creator, channel_id, relevance, video_count, sample_topics, sample_videos
ORDER BY relevance DESC
LIMIT $top_n
'''

EXAMPLE_VIDEOS_CYPHER = '''
CALL db.index.vector.queryNodes('youtube_comment_topic_embedding_index', $k, $q) YIELD node AS t, score
WHERE coalesce(t.platform, 'youtube') = $platform
MATCH (v:YouTubeVideo)-[rel:HAS_COMMENT_TOPIC]->(t)
WHERE coalesce(rel.platform, 'youtube') = $platform
WITH v, t, rel, score
ORDER BY score * coalesce(rel.weight, 1.0) DESC
WITH v,
     collect({topic: t.name, weight: rel.weight, score: score})[0..3] AS matches,
     max(score * coalesce(rel.weight, 1.0)) AS best
RETURN v.video_id AS video_id,
       v.video_title AS title,
       coalesce(v.channel_title, v.username) AS creator,
       v.video_url AS url,
       v.view_count AS views,
       best AS relevance,
       matches
ORDER BY best DESC
LIMIT $top_n
'''

COMMENT_DISCUSSIONS_CYPHER = '''
CALL db.index.vector.queryNodes('video_summary_embedding_index', $k, $q) YIELD node AS v, score
RETURN v.video_id AS video_id,
       v.video_title AS title,
       coalesce(v.channel_title, v.username) AS creator,
       coalesce(v.channel_id, '') AS channel_id,
       v.video_url AS url,
       score,
       v.comment_summary_description AS comment_summary_description
ORDER BY score DESC
LIMIT $top_n
'''

CONTENT_VIDEOS_CYPHER = '''
CALL db.index.vector.queryNodes('video_content_embedding_index', $k, $q) YIELD node AS v, score
RETURN v.video_id AS video_id,
       v.video_title AS title,
       coalesce(v.channel_title, v.username) AS creator,
       coalesce(v.channel_id, '') AS channel_id,
       v.video_url AS url,
       v.view_count AS views,
       v.thumbnail_url AS thumbnail_url,
       v.thumbnail_description AS thumbnail_description,
       v.video_description AS video_description,
       v.thumbnail_keywords AS thumbnail_keywords,
       v.tags AS tags,
       score
ORDER BY score DESC
LIMIT $top_n
'''

CONTENT_TOP_CREATORS_CYPHER = '''
CALL db.index.vector.queryNodes('video_content_embedding_index', $k, $q) YIELD node AS v, score
OPTIONAL MATCH (c:YouTubeChannel)-[:HAS_VIDEO]->(v)
WITH coalesce(c.channel_title, v.channel_title, v.username, 'unknown') AS creator,
     coalesce(c.channel_id, v.channel_id) AS channel_id,
     v, score
WITH creator, channel_id,
     sum(score) AS relevance,
     count(DISTINCT v) AS video_count,
     collect({video_id: v.video_id, title: v.video_title, url: v.video_url, score: score})[0..5] AS sample_videos
RETURN creator, channel_id, relevance, video_count, sample_videos
ORDER BY relevance DESC
LIMIT $top_n
'''


def top_creators(theme: str, top_n: int = 10, k: int = 200):
    q = embed_query(theme)
    with driver.session(database=NEO4J_DATABASE) as s:
        return [dict(r) for r in s.run(TOP_CREATORS_CYPHER, q=q, k=k, top_n=top_n, platform=YOUTUBE_PLATFORM)]

def example_videos(theme: str, top_n: int = 10, k: int = 100):
    q = embed_query(theme)
    with driver.session(database=NEO4J_DATABASE) as s:
        return [dict(r) for r in s.run(EXAMPLE_VIDEOS_CYPHER, q=q, k=k, top_n=top_n, platform=YOUTUBE_PLATFORM)]

def comment_discussions(theme: str, top_n: int = 10, k: int = 20):
    q = embed_query(theme)
    with driver.session(database=NEO4J_DATABASE) as s:
        return [dict(r) for r in s.run(COMMENT_DISCUSSIONS_CYPHER, q=q, k=max(k, top_n), top_n=top_n)]

def content_videos(theme: str, top_n: int = 10, k: int = 50):
    q = embed_query(theme)
    with driver.session(database=NEO4J_DATABASE) as s:
        return [dict(r) for r in s.run(CONTENT_VIDEOS_CYPHER, q=q, k=max(k, top_n), top_n=top_n)]

def content_top_creators(theme: str, top_n: int = 10, k: int = 200):
    q = embed_query(theme)
    with driver.session(database=NEO4J_DATABASE) as s:
        return [dict(r) for r in s.run(CONTENT_TOP_CREATORS_CYPHER, q=q, k=k, top_n=top_n, platform=YOUTUBE_PLATFORM)]


# --- Unified fused search ---------------------------------------------------
# Runs all three video-level indexes and fuses rankings with Reciprocal Rank
# Fusion (RRF). RRF is robust against differences in score scales between
# indexes and is the standard way to combine heterogeneous retrievers.

UNIFIED_CONTENT_Q = '''
CALL db.index.vector.queryNodes('video_content_embedding_index', $k, $q) YIELD node AS v, score
RETURN v.video_id AS video_id, score
'''

UNIFIED_SUMMARY_Q = '''
CALL db.index.vector.queryNodes('video_summary_embedding_index', $k, $q) YIELD node AS v, score
RETURN v.video_id AS video_id, score
'''

UNIFIED_TOPIC_Q = '''
CALL db.index.vector.queryNodes('youtube_comment_topic_embedding_index', $k, $q) YIELD node AS t, score
WHERE coalesce(t.platform, 'youtube') = $platform
MATCH (v:YouTubeVideo)-[rel:HAS_COMMENT_TOPIC]->(t)
WHERE coalesce(rel.platform, 'youtube') = $platform
WITH v.video_id AS video_id, max(score * coalesce(rel.weight, 1.0)) AS score
RETURN video_id, score
'''

UNIFIED_HYDRATE = '''
UNWIND $video_ids AS vid
MATCH (v:YouTubeVideo {video_id: vid})
OPTIONAL MATCH (c:YouTubeChannel)-[:HAS_VIDEO]->(v)
RETURN v.video_id AS video_id,
       v.video_title AS title,
       coalesce(c.channel_title, v.channel_title, v.username) AS creator,
       coalesce(c.channel_id, v.channel_id) AS channel_id,
       v.video_url AS url,
       v.view_count AS views,
       v.thumbnail_description AS thumbnail_description,
       v.video_description AS video_description,
       v.comment_summary_description AS comment_summary_description
'''


def _rrf_rank(rows, id_key: str = 'video_id', k_const: int = 60) -> Dict[str, float]:
    """rows are already sorted by descending score -- we only need the rank."""
    scores: Dict[str, float] = {}
    for rank, row in enumerate(rows):
        scores[row[id_key]] = 1.0 / (k_const + rank + 1)
    return scores

#pulls top matches from content, comment summaries, and topics
#converts each position into an RRF score and then sums those scores per video across the lists (assigns higher  importance to videos ranked highly in multiple lists based on their position, rather than raw similarity scores) 
#orders the videos by the sum of their RRF scores
#finally it returns the top n videos 
def unified_search(theme: str, top_n: int = 10, k_each: int = 50, include_creator_rollup: bool = True):
    q = embed_query(theme)
    with driver.session(database=NEO4J_DATABASE) as s:
        content_rows = [dict(r) for r in s.run(UNIFIED_CONTENT_Q, q=q, k=k_each)]
        summary_rows = [dict(r) for r in s.run(UNIFIED_SUMMARY_Q, q=q, k=k_each)]
        topic_rows   = [dict(r) for r in s.run(UNIFIED_TOPIC_Q,   q=q, k=k_each * 4, platform=YOUTUBE_PLATFORM)]

        content_rank = _rrf_rank(content_rows)
        summary_rank = _rrf_rank(summary_rows)
        topic_rank   = _rrf_rank(topic_rows)

        fused: Dict[str, float] = {}
        signals: Dict[str, List[str]] = {}
        for ranks, label in [(content_rank, 'content'), (summary_rank, 'comments'), (topic_rank, 'topic')]:
            for vid, s_val in ranks.items():
                fused[vid] = fused.get(vid, 0.0) + s_val
                signals.setdefault(vid, []).append(label)

        top_ids = sorted(fused.keys(), key=lambda x: fused[x], reverse=True)[:top_n]
        if not top_ids:
            return {'videos': [], 'creators': []}

        hydrated = {r['video_id']: dict(r) for r in s.run(UNIFIED_HYDRATE, video_ids=top_ids, platform=YOUTUBE_PLATFORM)}
        videos = []
        for vid in top_ids:
            if vid not in hydrated:
                continue
            row = hydrated[vid]
            row['fused_score'] = round(fused[vid], 5)
            row['matched_signals'] = signals[vid]
            videos.append(row)

        creators = []
        if include_creator_rollup:
            rollup: Dict[str, Dict[str, Any]] = {}
            for row in videos:
                cid = row['channel_id'] or row['creator'] or 'unknown'
                agg = rollup.setdefault(cid, {
                    'channel_id': cid,
                    'creator': row['creator'],
                    'relevance': 0.0,
                    'video_count': 0,
                    'videos': [],
                })
                agg['relevance'] += row['fused_score']
                agg['video_count'] += 1
                agg['videos'].append({
                    'video_id': row['video_id'],
                    'title': row['title'],
                    'url': row.get('url'),
                })
            creators = sorted(rollup.values(), key=lambda x: x['relevance'], reverse=True)

    return {'videos': videos, 'creators': creators}


## Similarity score analysis (production thresholds)

For each **query string**, run vector search against all **three** indexes (`video_content_embedding`, `comment_summary_embedding`, `topic_embedding`). For each index:

- print **query name** + index name
- show an **ASCII histogram** of cosine similarity `score` (20 bins on \[0, 1\], with spill counts outside that range)
- print up to **5 example rows** per score band: **0.90–1.00**, **0.80–0.89**, **0.70–0.79**, **0.60–0.69**, **0.50–0.59**

Tune `SIMILARITY_K_FRACTION` (default **0.1** = 10%), `SIMILARITY_K_MIN` / `SIMILARITY_K_MAX`, optional fixed override `SIMILARITY_ANALYSIS_K`, and `QUERIES_FOR_THRESHOLD_TUNING`, then run the next cell.

In [ ]:
import os
from pprint import pprint
from typing import Any, Dict, List, Optional, Tuple

SIMILARITY_K_FRACTION = float(os.getenv('SIMILARITY_K_FRACTION', '0.2'))
SIMILARITY_K_MIN = int(os.getenv('SIMILARITY_K_MIN', '50'))
SIMILARITY_K_MAX = int(os.getenv('SIMILARITY_K_MAX', '25000'))

# Optional: set to an integer string to force the same k for every index (overrides 10% sizing).
SIMILARITY_K_OVERRIDE_RAW = os.getenv('SIMILARITY_ANALYSIS_K', '').strip()

_INDEXED_NODE_COUNTS_CYPHER: Dict[str, str] = {
    'video_content_embedding_index': (
        'MATCH (v:YouTubeVideo) WHERE v.video_content_embedding IS NOT NULL RETURN count(v) AS c'
    ),
    'video_summary_embedding_index': (
        'MATCH (v:YouTubeVideo) WHERE v.comment_summary_embedding IS NOT NULL RETURN count(v) AS c'
    ),
    'youtube_comment_topic_embedding_index': (
        "MATCH (t:YouTubeCommentTopic) WHERE t.embedding IS NOT NULL AND coalesce(t.platform, 'youtube') = 'youtube' RETURN count(t) AS c"
    ),
}


def _indexed_node_counts() -> Dict[str, int]:
    out: Dict[str, int] = {}
    with driver.session(database=NEO4J_DATABASE) as s:
        for index_name, cy in _INDEXED_NODE_COUNTS_CYPHER.items():
            out[index_name] = int(s.run(cy).single()['c'])
    return out


def _k_for_index(index_name: str, counts: Dict[str, int]) -> int:
    if SIMILARITY_K_OVERRIDE_RAW:
        return max(SIMILARITY_K_MIN, int(SIMILARITY_K_OVERRIDE_RAW))
    n = counts.get(index_name, 0)
    k = int(SIMILARITY_K_FRACTION * n)
    return max(SIMILARITY_K_MIN, min(SIMILARITY_K_MAX, k))


# Edit this list for production threshold experiments
QUERIES_FOR_THRESHOLD_TUNING = [
    # 'imagery of weapons or violence',
    # 'vaping',
     'gambling',
    #  'mental health',
    # 'redbull',
    # "energy drinks",
]

CONTENT_SCORES_CYPHER = '''
CALL db.index.vector.queryNodes('video_content_embedding_index', $k, $q) YIELD node AS v, score
RETURN v.video_id AS video_id, v.video_title AS title, v.video_url AS url, score
'''

SUMMARY_SCORES_CYPHER = '''
CALL db.index.vector.queryNodes('video_summary_embedding_index', $k, $q) YIELD node AS v, score
RETURN v.video_id AS video_id, v.video_title AS title, v.video_url AS url, score
'''

TOPIC_SCORES_CYPHER = '''
CALL db.index.vector.queryNodes('youtube_comment_topic_embedding_index', $k, $q) YIELD node AS t, score
WHERE coalesce(t.platform, 'youtube') = 'youtube'
MATCH (v:YouTubeVideo)-[rel:HAS_COMMENT_TOPIC]->(t)
WHERE coalesce(rel.platform, 'youtube') = 'youtube'
RETURN t.name AS topic_name,
       v.video_id AS video_id, v.video_title AS title, v.video_url AS url,
       score,
       rel.weight AS topic_weight,
       rel.position AS topic_position,
       score * coalesce(rel.weight, 1.0) AS weighted_similarity
'''

# (label, low inclusive, high exclusive) except top bucket high is inclusive via next bucket start
SCORE_BANDS: List[Tuple[str, float, float]] = [
    ('0.90–1.00', 0.9, 1.0000001),
    ('0.80–0.89', 0.8, 0.9),
    ('0.70–0.79', 0.7, 0.8),
    ('0.60–0.69', 0.6, 0.7),
    ('0.50–0.59', 0.5, 0.6),
]


def _fetch_rows(cypher: str, q: List[float], k: int) -> List[Dict[str, Any]]:
    with driver.session(database=NEO4J_DATABASE) as s:
        return [dict(r) for r in s.run(cypher, q=q, k=k)]


def _scores(rows: List[Dict[str, Any]]) -> List[float]:
    return [float(r['score']) for r in rows if r.get('score') is not None]


def _ascii_histogram(scores: List[float], n_bins: int = 20, width: int = 50) -> None:
    if not scores:
        print('  (no scores)')
        return
    lo, hi = min(scores), max(scores)
    # Bin into [0,1] plus spill for outliers
    lo_b, hi_b = 0.0, 1.0
    counts = [0] * n_bins
    under = over = 0
    for s in scores:
        if s < lo_b:
            under += 1
        elif s > hi_b:
            over += 1
        else:
            idx = int((s - lo_b) / (hi_b - lo_b) * n_bins)
            if idx >= n_bins:
                idx = n_bins - 1
            counts[idx] += 1
    mx = max(counts) if counts else 1
    print(f'  score min={lo:.4f} max={hi:.4f}  (binning [0,1] with {n_bins} bins; below0={under} above1={over})')
    for i, c in enumerate(counts):
        left = lo_b + (hi_b - lo_b) * i / n_bins
        right = lo_b + (hi_b - lo_b) * (i + 1) / n_bins
        bar = '#' * max(1, int(width * c / mx)) if c else ''
        print(f'  [{left:.2f},{right:.2f}): {c:4d}  {bar}')


def _band_examples(rows: List[Dict[str, Any]], label: str, lo: float, hi: float, n: int = 5) -> None:
    band = [r for r in rows if r.get('score') is not None and lo <= float(r['score']) < hi]
    band.sort(key=lambda r: float(r['score']), reverse=True)
    print(f'  band {label}: {len(band)} hits (showing top {min(n, len(band))})')
    for r in band[:n]:
        pprint(r)


def print_topic_retriever_topic_name_matches(
    rows: List[Dict[str, Any]],
    topic_name_substring: Optional[str] = None,
) -> None:
    """Print topic retriever rows whose topic_name contains topic_name_substring (case-insensitive).

    Does not call the embedding model or Neo4j — pass rows from an existing TOPIC_SCORES_CYPHER fetch.
    If topic_name_substring is None or blank, prints nothing.
    """
    if not topic_name_substring or not str(topic_name_substring).strip():
        return
    needle = str(topic_name_substring).strip().lower()
    matched = [r for r in rows if needle in (r.get('topic_name') or '').lower()]
    if not matched:
        return
    matched.sort(
        key=lambda r: float(
            r.get('weighted_similarity')
            if r.get('weighted_similarity') is not None
            else r.get('score') or 0.0
        ),
        reverse=True,
    )
    print('-' * 88)
    print(f'  TOPIC ROWS WITH topic_name containing {topic_name_substring.strip()!r}  (n={len(matched)})')
    for r in matched:
        pprint(r)


def analyze_similarity_for_query(
    query_name: str,
    counts: Dict[str, int],
    *,
    optional_topic_name_substring: Optional[str] = None,
) -> None:
    print('=' * 88)
    print(
        f'QUERY: {query_name!r}   (k ≈ {SIMILARITY_K_FRACTION * 100:.0f}% of indexed nodes per index, '
        f'min={SIMILARITY_K_MIN} max={SIMILARITY_K_MAX}'
        + (f", override={SIMILARITY_K_OVERRIDE_RAW!r}" if SIMILARITY_K_OVERRIDE_RAW else '')
        + ')'
    )
    print('=' * 88)
    q = embed_query(query_name)

    suites: List[Tuple[str, str]] = [
        ('video_content_embedding_index', CONTENT_SCORES_CYPHER),
        ('video_summary_embedding_index', SUMMARY_SCORES_CYPHER),
        ('youtube_comment_topic_embedding_index', TOPIC_SCORES_CYPHER),
    ]

    for index_name, cypher in suites:
        k = _k_for_index(index_name, counts)
        rows = _fetch_rows(cypher, q, k)
        scores = _scores(rows)
        print('-' * 88)
        n_idx = counts.get(index_name, 0)
        print(f'INDEX: {index_name}   indexed_nodes={n_idx}   k_used={k}   rows_returned={len(rows)}')
        _ascii_histogram(scores)
        for band_label, lo, hi in SCORE_BANDS:
            _band_examples(rows, band_label, lo, hi, n=5)
        if index_name == 'youtube_comment_topic_embedding_index' and optional_topic_name_substring:
            print_topic_retriever_topic_name_matches(rows, optional_topic_name_substring)
        print()


_COUNTS_GLOBAL = _indexed_node_counts()
print('Indexed node counts (used to size k per index):', _COUNTS_GLOBAL)

for _q in QUERIES_FOR_THRESHOLD_TUNING:
    analyze_similarity_for_query(_q, counts=_COUNTS_GLOBAL)


In [ ]:
from pprint import pprint

THEME = 'imagery of weapons or violence'
TOP_N = 10

print(f'THEME: {THEME}')
print(f'TOP_N: {TOP_N}')

In [ ]:
print(f'--- CONTENT: videos that visually / textually depict "{THEME}" ---')
for row in content_videos(THEME, top_n=TOP_N):
    pprint(row)

In [ ]:
print(f'--- CONTENT: creators whose videos depict "{THEME}" ---')
for row in content_top_creators(THEME, top_n=TOP_N):
    pprint(row)

In [ ]:
print(f'--- TOPICS: creators whose comment sections discuss "{THEME}" ---')
for row in top_creators(THEME, top_n=TOP_N):
    pprint(row)

In [ ]:
print(f'--- COMMENT SUMMARIES matching "{THEME}" ---')
for row in comment_discussions(THEME, top_n=TOP_N):
    pprint(row)

In [ ]:
print(f'--- UNIFIED fused search for "{THEME}" ---')
fused = unified_search(THEME, top_n=TOP_N)
print('Top videos (with which signals matched):')
for row in fused['videos']:
    pprint(row)

In [ ]:
print(f'--- UNIFIED creator rollup for "{THEME}" ---')
fused = unified_search(THEME, top_n=TOP_N)
for row in fused['creators']:
    pprint(row)

In [ ]:
try:
    driver.close()
except Exception:
    pass
try:
    mongo_client.close()
except Exception:
    pass
print('Closed clients.')